# SQL Fundamentals

This notebook covers the essential SQL concepts: DDL, DML, DQL, and TCL statements.

> **Note:** Since Jupyter Notebooks don't natively support SQL, we use the `%%sql` magic command. This requires loading the SQL extension first (`%load_ext sql`) and connecting to a DuckDB database. Each SQL cell must start with `%%sql` to be interpreted as SQL rather than Python.

## Setup

In [1]:
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Connect to in-memory database
%sql duckdb:///:memory:

## 1. SQL Statement Types Overview

SQL statements are categorized into four types:

| Category | Commands | Purpose |
|----------|----------|----------|
| **DDL** | CREATE, ALTER, DROP, TRUNCATE | Define/modify structure |
| **DML** | INSERT, UPDATE, DELETE | Manipulate data |
| **DQL** | SELECT | Query data |
| **TCL** | BEGIN, COMMIT, ROLLBACK | Control transactions |

## 2. DDL - Data Definition Language

### 2.1 CREATE TABLE - Basic

In [2]:
%%sql
-- Create a simple employees table (OR REPLACE for re-runnability)
CREATE OR REPLACE TABLE employees (
    id INTEGER PRIMARY KEY,
    name VARCHAR NOT NULL,
    department VARCHAR,
    salary INTEGER,
    hire_date DATE
);

-- Verify table structure
DESCRIBE employees;

,Success


### 2.2 CREATE TABLE with Constraints

In [3]:
%%sql
-- Create table with various constraints
CREATE OR REPLACE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    email VARCHAR UNIQUE NOT NULL,
    name VARCHAR NOT NULL,
    age INTEGER CHECK (age >= 18),
    balance DECIMAL(10, 2) DEFAULT 0.00,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

DESCRIBE customers;

,Success


### 2.3 CREATE TABLE AS (CTAS) - From Query

In [4]:
%%sql
-- Create table from query results
CREATE OR REPLACE TABLE sample_data AS
SELECT 
    range AS id,
    'User ' || range AS username,
    'user' || range || '@example.com' AS email
FROM range(5);

SELECT * FROM sample_data;

,id,username,email
0,0,User 0,user0@example.com
1,1,User 1,user1@example.com
2,2,User 2,user2@example.com
3,3,User 3,user3@example.com
4,4,User 4,user4@example.com


### 2.4 CREATE OR REPLACE TABLE

In [5]:
%%sql
-- Safely recreate table (useful in notebooks!)
CREATE OR REPLACE TABLE config (
    key VARCHAR,
    value VARCHAR
);

INSERT INTO config VALUES ('setting1', 'value1');
SELECT * FROM config;

,key,value
0,setting1,value1


### 2.5 ALTER TABLE

In [6]:
%%sql
-- Add column
ALTER TABLE employees ADD COLUMN is_active BOOLEAN DEFAULT TRUE;

-- Rename column
ALTER TABLE employees RENAME COLUMN name TO full_name;

DESCRIBE employees;

,Success


### 2.6 DROP and TRUNCATE

In [7]:
%%sql
-- TRUNCATE removes all data but keeps structure
TRUNCATE TABLE sample_data;
SELECT COUNT(*) AS rows_after_truncate FROM sample_data;

,rows_after_truncate
0,0


In [8]:
%%sql
-- DROP removes the entire table
DROP TABLE IF EXISTS sample_data;
DROP TABLE IF EXISTS config;

,Success


## 3. Data Types in DuckDB

### 3.1 Numeric Types

In [9]:
%%sql
SELECT 
    42::INTEGER AS integer_num,
    9223372036854775807::BIGINT AS big_int,
    123.45::DECIMAL(10,2) AS decimal_num,
    3.14159::DOUBLE AS double_num;

,integer_num,big_int,decimal_num,double_num
0,42,9223372036854775807,123.45,3.14159


### 3.2 String and Boolean Types

In [10]:
%%sql
SELECT 
    'Hello DuckDB'::VARCHAR AS text,
    TRUE::BOOLEAN AS is_active,
    uuid() AS unique_id;

,text,is_active,unique_id
0,Hello DuckDB,True,cd6c8615-155b-4403-b0c4-03f4f6f70db8


### 3.3 Date and Time Types

In [11]:
%%sql
SELECT 
    DATE '2024-01-15' AS date_field,
    TIME '14:30:00' AS time_field,
    TIMESTAMP '2024-01-15 14:30:00' AS timestamp_field,
    INTERVAL '3 days' AS interval_field,
    CURRENT_DATE AS today,
    CURRENT_TIMESTAMP AS now;

,date_field,time_field,timestamp_field,interval_field,today,now
0,2024-01-15,14:30:00,2024-01-15 14:30:00,3 days,2025-11-27,2025-11-27 21:17:39.491561+01:00


### 3.4 Complex Types (DuckDB Special!)

In [12]:
%%sql
SELECT 
    [1, 2, 3, 4, 5] AS number_list,
    ['apple', 'banana', 'cherry'] AS string_list,
    {'name': 'John', 'age': 30} AS struct_data;

,number_list,string_list,struct_data
0,"[1, 2, 3, 4, 5]","[apple, banana, cherry]","{'name': 'John', 'age': 30}"


## 4. DML - Data Manipulation

### 4.1 INSERT - Adding Data

In [13]:
%%sql
-- Insert single row
INSERT INTO employees VALUES 
    (1, 'Alice Johnson', 'Engineering', 75000, '2020-01-15', TRUE);

-- Insert multiple rows
INSERT INTO employees VALUES 
    (2, 'Bob Smith', 'Sales', 60000, '2019-03-20', TRUE),
    (3, 'Charlie Brown', 'Engineering', 80000, '2021-06-01', TRUE),
    (4, 'Diana Prince', 'Marketing', 65000, '2020-09-10', TRUE),
    (5, 'Eve Davis', 'Engineering', 72000, '2022-02-14', TRUE),
    (6, 'Frank Miller', 'Sales', 58000, '2021-11-05', TRUE),
    (7, 'Grace Lee', 'HR', 55000, '2019-07-22', TRUE),
    (8, 'Henry Wilson', 'Engineering', 78000, '2020-12-01', FALSE),
    (9, 'Ivy Chen', 'Marketing', 62000, '2021-04-17', TRUE),
    (10, 'Jack Turner', 'Sales', 64000, '2022-08-30', TRUE);

SELECT * FROM employees;

,id,full_name,department,salary,hire_date,is_active
0,1,Alice Johnson,Engineering,75000,2020-01-15,True
1,2,Bob Smith,Sales,60000,2019-03-20,True
2,3,Charlie Brown,Engineering,80000,2021-06-01,True
3,4,Diana Prince,Marketing,65000,2020-09-10,True
4,5,Eve Davis,Engineering,72000,2022-02-14,True
5,6,Frank Miller,Sales,58000,2021-11-05,True
6,7,Grace Lee,HR,55000,2019-07-22,True
7,8,Henry Wilson,Engineering,78000,2020-12-01,False
8,9,Ivy Chen,Marketing,62000,2021-04-17,True
9,10,Jack Turner,Sales,64000,2022-08-30,True


### 4.2 UPDATE - Modifying Data

In [14]:
%%sql
-- Update single column
UPDATE employees
SET salary = salary * 1.05
WHERE department = 'Engineering';

-- Verify
SELECT full_name, department, salary 
FROM employees 
WHERE department = 'Engineering';

,full_name,department,salary
0,Alice Johnson,Engineering,78750
1,Charlie Brown,Engineering,84000
2,Eve Davis,Engineering,75600
3,Henry Wilson,Engineering,81900


### 4.3 DELETE - Removing Data

In [15]:
%%sql
-- Delete inactive employees
DELETE FROM employees WHERE is_active = FALSE;

SELECT COUNT(*) AS remaining_employees FROM employees;

,remaining_employees
0,9


## 5. SELECT - Querying Data

### 5.1 Basic SELECT

In [16]:
%%sql
-- Select all columns
SELECT * FROM employees LIMIT 5;

,id,full_name,department,salary,hire_date,is_active
0,1,Alice Johnson,Engineering,78750,2020-01-15,True
1,2,Bob Smith,Sales,60000,2019-03-20,True
2,3,Charlie Brown,Engineering,84000,2021-06-01,True
3,4,Diana Prince,Marketing,65000,2020-09-10,True
4,5,Eve Davis,Engineering,75600,2022-02-14,True


In [17]:
%%sql
-- Select specific columns
SELECT full_name, department, salary FROM employees;

,full_name,department,salary
0,Alice Johnson,Engineering,78750
1,Bob Smith,Sales,60000
2,Charlie Brown,Engineering,84000
3,Diana Prince,Marketing,65000
4,Eve Davis,Engineering,75600
5,Frank Miller,Sales,58000
6,Grace Lee,HR,55000
7,Ivy Chen,Marketing,62000
8,Jack Turner,Sales,64000


### 5.2 Column Aliases and Calculations

In [18]:
%%sql
SELECT 
    full_name AS employee,
    salary AS annual_salary,
    ROUND(salary / 12, 2) AS monthly_salary,
    salary * 1.1 AS salary_with_10pct_raise
FROM employees
LIMIT 5;

,employee,annual_salary,monthly_salary,salary_with_10pct_raise
0,Alice Johnson,78750,6562.50,86625.0
1,Bob Smith,60000,5000.00,66000.0
2,Charlie Brown,84000,7000.00,92400.0
3,Diana Prince,65000,5416.67,71500.0
4,Eve Davis,75600,6300.00,83160.0


### 5.3 DISTINCT - Remove Duplicates

In [19]:
%%sql
-- Get unique departments
SELECT DISTINCT department FROM employees ORDER BY department;

,department
0,Engineering
1,HR
2,Marketing
3,Sales


## 6. WHERE - Filtering Data

### 6.1 Comparison Operators

In [20]:
%%sql
-- Salary greater than 70000
SELECT full_name, department, salary 
FROM employees 
WHERE salary > 70000;

,full_name,department,salary
0,Alice Johnson,Engineering,78750
1,Charlie Brown,Engineering,84000
2,Eve Davis,Engineering,75600


In [21]:
%%sql
-- Equal comparison
SELECT full_name, salary 
FROM employees 
WHERE department = 'Engineering';

,full_name,salary
0,Alice Johnson,78750
1,Charlie Brown,84000
2,Eve Davis,75600


### 6.2 Logical Operators (AND, OR, NOT)

In [22]:
%%sql
-- AND: Both conditions must be true
SELECT full_name, department, salary 
FROM employees 
WHERE department = 'Engineering' AND salary > 75000;

,full_name,department,salary
0,Alice Johnson,Engineering,78750
1,Charlie Brown,Engineering,84000
2,Eve Davis,Engineering,75600


In [23]:
%%sql
-- OR: At least one condition true
SELECT full_name, department 
FROM employees 
WHERE department = 'Sales' OR department = 'Marketing';

,full_name,department
0,Bob Smith,Sales
1,Diana Prince,Marketing
2,Frank Miller,Sales
3,Ivy Chen,Marketing
4,Jack Turner,Sales


### 6.3 BETWEEN, IN, LIKE

In [24]:
%%sql
-- BETWEEN: Range (inclusive)
SELECT full_name, salary 
FROM employees 
WHERE salary BETWEEN 60000 AND 75000;

,full_name,salary
0,Bob Smith,60000
1,Diana Prince,65000
2,Ivy Chen,62000
3,Jack Turner,64000


In [25]:
%%sql
-- IN: Multiple values
SELECT full_name, department 
FROM employees 
WHERE department IN ('Sales', 'Marketing', 'HR');

,full_name,department
0,Bob Smith,Sales
1,Diana Prince,Marketing
2,Frank Miller,Sales
3,Grace Lee,HR
4,Ivy Chen,Marketing
5,Jack Turner,Sales


In [26]:
%%sql
-- LIKE: Pattern matching (% = any characters, _ = single character)
SELECT full_name FROM employees WHERE full_name LIKE 'A%';  -- Starts with A

,full_name
0,Alice Johnson


In [27]:
%%sql
SELECT full_name FROM employees WHERE full_name LIKE '%son%';  -- Contains 'son'

,full_name
0,Alice Johnson


### 6.4 IS NULL / IS NOT NULL

In [28]:
%%sql
-- Add employee with NULL department
INSERT INTO employees VALUES (11, 'Karen White', NULL, 50000, '2023-01-15', TRUE);

-- Find employees without department
SELECT full_name, department FROM employees WHERE department IS NULL;

,full_name,department
0,Karen White,None


## 7. ORDER BY - Sorting Results

In [29]:
%%sql
-- Ascending (default)
SELECT full_name, salary 
FROM employees 
WHERE department IS NOT NULL
ORDER BY salary ASC
LIMIT 5;

,full_name,salary
0,Grace Lee,55000
1,Frank Miller,58000
2,Bob Smith,60000
3,Ivy Chen,62000
4,Jack Turner,64000


In [30]:
%%sql
-- Descending
SELECT full_name, salary 
FROM employees 
WHERE department IS NOT NULL
ORDER BY salary DESC
LIMIT 5;

,full_name,salary
0,Charlie Brown,84000
1,Alice Johnson,78750
2,Eve Davis,75600
3,Diana Prince,65000
4,Jack Turner,64000


In [31]:
%%sql
-- Multiple columns
SELECT full_name, department, salary 
FROM employees 
WHERE department IS NOT NULL
ORDER BY department ASC, salary DESC;

,full_name,department,salary
0,Charlie Brown,Engineering,84000
1,Alice Johnson,Engineering,78750
2,Eve Davis,Engineering,75600
3,Grace Lee,HR,55000
4,Diana Prince,Marketing,65000
5,Ivy Chen,Marketing,62000
6,Jack Turner,Sales,64000
7,Bob Smith,Sales,60000
8,Frank Miller,Sales,58000


### LIMIT and OFFSET (Pagination)

In [32]:
%%sql
-- Skip first 3, get next 3
SELECT full_name, salary 
FROM employees 
WHERE department IS NOT NULL
ORDER BY salary DESC 
LIMIT 3 OFFSET 3;

,full_name,salary
0,Diana Prince,65000
1,Jack Turner,64000
2,Ivy Chen,62000


## 8. Aggregate Functions

Aggregate functions calculate a single result from multiple rows.

In [33]:
%%sql
SELECT 
    COUNT(*) AS total_employees,
    COUNT(DISTINCT department) AS unique_departments,
    SUM(salary) AS total_payroll,
    ROUND(AVG(salary), 2) AS average_salary,
    MIN(salary) AS min_salary,
    MAX(salary) AS max_salary
FROM employees
WHERE department IS NOT NULL;

,total_employees,unique_departments,total_payroll,average_salary,min_salary,max_salary
0,9,4,602350.0,66927.78,55000,84000


## 9. GROUP BY - Grouping Data

### 9.1 Basic Grouping

In [34]:
%%sql
-- Count employees per department
SELECT 
    department,
    COUNT(*) AS employee_count,
    ROUND(AVG(salary), 2) AS avg_salary,
    MIN(salary) AS min_salary,
    MAX(salary) AS max_salary
FROM employees
WHERE department IS NOT NULL
GROUP BY department
ORDER BY avg_salary DESC;

,department,employee_count,avg_salary,min_salary,max_salary
0,Engineering,3,79450.00,75600,84000
1,Marketing,2,63500.00,62000,65000
2,Sales,3,60666.67,58000,64000
3,HR,1,55000.00,55000,55000


### 9.2 HAVING - Filter Groups

**WHERE** filters rows before grouping, **HAVING** filters groups after aggregation.

In [35]:
%%sql
-- Only departments with more than 2 employees
SELECT 
    department,
    COUNT(*) AS employee_count,
    ROUND(AVG(salary), 2) AS avg_salary
FROM employees
WHERE department IS NOT NULL
GROUP BY department
HAVING COUNT(*) > 2
ORDER BY employee_count DESC;

,department,employee_count,avg_salary
0,Engineering,3,79450.00
1,Sales,3,60666.67


In [36]:
%%sql
-- Departments with average salary > 65000
SELECT 
    department,
    COUNT(*) AS employee_count,
    ROUND(AVG(salary), 2) AS avg_salary
FROM employees
WHERE department IS NOT NULL
GROUP BY department
HAVING AVG(salary) > 65000;

,department,employee_count,avg_salary
0,Engineering,3,79450.0


### 9.3 Query Execution Order

Understanding the order SQL processes clauses:

```
FROM        → 1. Get data from tables
WHERE       → 2. Filter individual rows
GROUP BY    → 3. Group rows
HAVING      → 4. Filter groups
SELECT      → 5. Select columns
ORDER BY    → 6. Sort results
LIMIT       → 7. Limit results
```

## 10. Transactions

> **Note:** JupySQL's `%%sql` magic automatically manages transactions. Each cell runs in autocommit mode. For explicit transaction control, you would use the Python API directly. The examples below show the SQL syntax for reference.

In [37]:
%%sql
-- Simulate a "transfer" between employees (runs in autocommit)
UPDATE employees SET salary = salary - 1000 WHERE id = 1;
UPDATE employees SET salary = salary + 1000 WHERE id = 2;

SELECT full_name, salary FROM employees WHERE id IN (1, 2);

,full_name,salary
0,Alice Johnson,77750
1,Bob Smith,61000


In [38]:
# For explicit transaction control, use Python API:
import duckdb

conn = duckdb.connect(':memory:')
conn.execute("CREATE TABLE test_tx (id INT, val INT)")
conn.execute("INSERT INTO test_tx VALUES (1, 100)")

# Start transaction, make change, then rollback
conn.begin()
conn.execute("UPDATE test_tx SET val = 999 WHERE id = 1")
conn.rollback()  # Undo the change

result = conn.execute("SELECT * FROM test_tx").fetchall()
print(f"After ROLLBACK: {result}")  # Still (1, 100)
conn.close()

After ROLLBACK: [(1, 100)]


## 11. Views and Indexes

### 11.1 CREATE VIEW

In [39]:
%%sql
-- Create a view (virtual table)
CREATE OR REPLACE VIEW high_earners AS
SELECT 
    full_name,
    department,
    salary,
    'Premium' AS tier
FROM employees
WHERE salary > 70000;

-- Query the view like a table
SELECT * FROM high_earners;

,full_name,department,salary,tier
0,Alice Johnson,Engineering,77750,Premium
1,Charlie Brown,Engineering,84000,Premium
2,Eve Davis,Engineering,75600,Premium


### 11.2 CREATE INDEX

In [40]:
%%sql
-- Drop and recreate index (IF NOT EXISTS not supported for indexes in DuckDB)
DROP INDEX IF EXISTS idx_employees_dept;
CREATE INDEX idx_employees_dept ON employees(department);

SELECT 'Index created!' AS status;

,status
0,Index created!


## 🎯 Practice Exercises

Try solving these yourself:

1. Find all employees hired after 2021-01-01
2. List the 3 lowest-paid employees
3. Find departments where the highest salary exceeds 75000
4. Count how many employees have 'e' in their name
5. Calculate the salary range (max - min) for each department

In [41]:
%%sql
-- Your practice queries here

UnboundLocalError: cannot access local variable 'result' where it is not associated with a value

## Clean Up

In [ ]:
%%sql
DROP VIEW IF EXISTS high_earners;
DROP TABLE IF EXISTS employees;
DROP TABLE IF EXISTS customers;

RuntimeError: (_duckdb.TransactionException) TransactionContext Error: Current transaction is aborted (please ROLLBACK)
[SQL: DROP VIEW IF EXISTS high_earners;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


## 📚 Summary

### SQL Statement Categories

| Category | Commands | Purpose |
|----------|----------|----------|
| **DDL** | CREATE, ALTER, DROP, TRUNCATE | Define/modify structure |
| **DML** | INSERT, UPDATE, DELETE | Manipulate data |
| **DQL** | SELECT | Query data |
| **TCL** | BEGIN, COMMIT, ROLLBACK | Control transactions |

### Key Concepts

- **WHERE** filters individual rows (before grouping)
- **HAVING** filters groups (after aggregation)
- **ORDER BY** sorts results (ASC/DESC)
- **LIMIT/OFFSET** for pagination
- **DISTINCT** removes duplicate rows

### Common Data Types

- `INTEGER`, `BIGINT`, `DECIMAL` - Numbers
- `VARCHAR`, `TEXT` - Strings
- `DATE`, `TIMESTAMP` - Date/Time
- `BOOLEAN` - True/False
- `LIST`, `STRUCT` - Complex types (DuckDB special)